# Ollama Server on Google Colab

Run this notebook in Google Colab with **GPU** and **Internet** enabled.

This notebook starts one Ollama server and uses it for both `embeddinggemma` and `qwen3:4b`.

In [ ]:
# Install Ollama
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
# Start Ollama
import os, subprocess, time, requests

env = os.environ.copy()
env['OLLAMA_HOST'] = '127.0.0.1:11434'
log = open('/tmp/ollama.log', 'a')
ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT
)

for _ in range(30):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('Ollama did not start. Check /tmp/ollama.log')

print('Ollama is running at http://127.0.0.1:11434')

In [ ]:
# Download both models
!ollama pull embeddinggemma
!ollama pull qwen3:4b
!ollama list

In [ ]:
# Test both APIs
BASE_URL = 'http://127.0.0.1:11434'

embedding = requests.post(
    f'{BASE_URL}/api/embed',
    json={'model': 'embeddinggemma', 'input': 'Hello from Colab'},
    timeout=120
)
embedding.raise_for_status()
print('Embedding size:', len(embedding.json()['embeddings'][0]))

chat = requests.post(
    f'{BASE_URL}/api/generate',
    json={
        'model': 'qwen3:4b',
        'prompt': 'Explain Ollama in one sentence.',
        'stream': False
    },
    timeout=120
)
chat.raise_for_status()
print(chat.json()['response'])

## Optional: expose the same server with Pinggy

Run the next cell, then type the SSH command in the terminal that appears. Copy the HTTPS URL printed by Pinggy and use it as `BASE_URL` in another notebook.

Keep this Colab runtime and terminal running while you use the URL.

In [ ]:
%pip -q install colab-xterm
%load_ext colabxterm
%xterm

In the terminal, run:

```bash
ssh -p 443 -R0:localhost:11434 -t qr@free.pinggy.io "u:Host:localhost:11434"
```

Both models are available through the same public URL. Select the model in the JSON request using the `model` field. Do not share the URL publicly; Ollama has no authentication by default.

> This file is generated by ChatGPT - GPT 5.6 Luna